# Import packages and data

In [33]:
import pandas as pd
import numpy as np

def normalize_to_62(phone_series):
    # 1. Convert to string and strip invisible spaces from the edges
    cleaned = phone_series.astype(str).str.strip()
    
    # 2. Fix the float '.0' issue
    cleaned = cleaned.str.replace(r'\.0$', '', regex=True)
    
    def fix_prefix(x):
        # 🛡️ Force x to be a string just in case pandas sneaks a float in!
        x_str = str(x).strip()
        
        # Ignore empty values
        if x_str.lower() in ['nan', 'none', '<na>', '']:
            return np.nan 
            
        # If it starts with '0', replace '0' with '62'
        if x_str.startswith('0'):
            return '62' + x_str[1:]
            
        # If it starts with '8', prepend '62'
        elif x_str.startswith('8'):
            return '62' + x_str
            
        return x_str

    return cleaned.apply(fix_prefix)



In [26]:
# 1. Load datasets 
data_1 = pd.read_csv('../storage/seller_candidates_20260418_010245.csv')
data_2 = pd.read_csv('../storage/seller_candidates_20260419_014612.csv')
#existing_data = pd.read_csv('../storage/master_database.csv')



# Inspect and normalize data

In [6]:
# Display data in scrollable format
from IPython.display import display, HTML

# Set pandas display options for scrollable table
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# Display as scrollable HTML table
display(HTML(f'<div style="max-height: 400px; overflow-y: scroll;">{data_1.to_html()}</div>'))

# Alternative: Use pandas Styler for better formatting
# data_1.style.set_properties(**{'max-width': '100px', 'text-overflow': 'ellipsis', 'overflow': 'hidden'})

In [ ]:
# 2. Force phone numbers to be text (strings)
# If the data already in 628xxx format
data_1['phone'] = data_1['phone'].apply(lambda x: f"{x:.0f}" if pd.notnull(x) else np.nan)
# If the data in various formats, convert to 62 format
data_2['phone'] = normalize_to_62(data_2['phone'])

#existing_db['phone'] = existing_db['phone'].astype(str).str.replace('\.0', '', regex=True)


In [39]:
print(data_2['phone'])

0               NaN
1               NaN
2               NaN
3               NaN
4               NaN
5               NaN
6               NaN
7               NaN
8               NaN
9               NaN
10              NaN
11              NaN
12    6281253253523
13              NaN
14              NaN
15              NaN
16              NaN
17              NaN
18    6282194683040
19              NaN
20    6282259124978
21              NaN
22              NaN
23    6281347707003
24              NaN
25              NaN
26              NaN
27              NaN
28              NaN
29              NaN
30    6282228837501
31    6282394467641
32              NaN
33    6282196902205
34              NaN
35    6281399944777
36              NaN
37    6282228837501
38              NaN
39              NaN
40              NaN
41              NaN
42    6281399944777
43              NaN
44    6281347707003
45              NaN
46              NaN
47              NaN
48              NaN
49              NaN


In [18]:
# Display as scrollable HTML table
display(HTML(f'<div style="max-height: 400px; overflow-y: scroll;">{data_2.to_html()}</div>'))

# Merge all scraped data
Merge scraped data as one and drop the NaN/empty value in phone

In [41]:
combined_data = pd.concat([data_1, data_2], ignore_index=True)

# 2. STANDARDIZE EMPTY VALUES: 
# Make sure phone numbers are strings, strip hidden spaces, and lowercase it
combined_data['phone'] = combined_data['phone'].astype(str).str.strip().str.lower()

# Turn all the fake empty text (like "nan", "none", or just "") into REAL missing data (np.nan)
combined_data['phone'] = combined_data['phone'].replace(['nan', 'none', '<na>', ''], np.nan)

# 3. DROP THE ROWS: Delete any row where 'phone' is exactly np.nan
final_df = combined_data.dropna(subset=['phone'])

# (Optional) Reset the index so your row numbers count perfectly from 0 to the end
final_df = final_df.reset_index(drop=True)

print(f"Total rows after merging and cleaning: {len(final_df)}")
print(final_df['phone'])

Total rows after merging and cleaning: 90
0      6285768582236
1      6285783882272
2      6285783882272
3      6285789496358
4      6283845477712
5       628971254732
6      6288268038302
7      6282364019423
8      6285832004776
9      6282228837501
10     6289502165372
11     6282189305706
12     6285138681033
13     6285767813923
14     6283187930695
15     6283845269848
16    62895337758607
17     6283137983127
18     6289673914101
19     6285609915074
20     6281317671803
21     6283871105155
22     6281340886109
23     6282164444569
24     6285381153914
25     6285783563025
26     6285764698170
27     6285764698170
28     6285832005299
29     6283192696977
30     6285835503247
31     6282189853291
32     6282189853291
33    62887437064890
34     6287716922788
35     6282196902205
36     6289518051382
37     6285255405454
38     6282228837501
39     6285342911394
40     6285166523328
41     6285166523328
42     6282189305706
43     6285242747149
44     6282349931627
45     628387

# Compare scraped data with master data

In [ ]:


# 3. The Magic Merge
# We merge based on the phone number. 
# indicator=True creates a new column called '_merge' that tells us where the data came from.
comparison = new_scrape.merge(
    existing_db[['phone', 'registered_name']], # Only bring in specific columns from the master DB to keep it clean
    on='phone', 
    how='left', 
    indicator=True
)

# 4. Split the data based on the results
brand_new_sellers = comparison[comparison['_merge'] == 'left_only']
already_registered = comparison[comparison['_merge'] == 'both']

print(f"Total Scraped Today: {len(new_scrape)}")
print(f"Brand New Leads: {len(brand_new_sellers)}")
print(f"Already Registered: {len(already_registered)}")

In [ ]:
# Show sellers where the Facebook Author Name doesn't match our Registered Name
name_mismatch = already_registered[already_registered['author_name'] != already_registered['registered_name']]

display(name_mismatch[['phone', 'author_name', 'registered_name', 'post_text']])